# SocialPulse Market Intelligence - Exploratory Data Analysis

**Objective:** identify audience interests, engagement patterns, and emerging topics across social platforms to inform marketing strategy.

**Data sources & scope:**
- `unified_posts` (YouTube + Instagram) - balanced across all 12 keywords, recent, on-topic, used for analysis and modeling.
- `twitter_eda` (Twitter) - kept separate because it is ~99% AWS and pre-ChatGPT, used for descriptive EDA only, never modeling, to avoid bias contamination.

Engagement Score is a raw cross-platform proxy (YouTube: likes; Instagram: likes+comments; Twitter: likes+retweets) and is not directly comparable across platforms, so compare within a platform.

In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

DATA = Path("../data/clean")
REPORT_DIR = Path("../data/reports/eda")
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'pandas'

## 1. Load data

In [ ]:
unified = pd.read_csv(DATA / "unified_dataset.csv")
twitter = pd.read_csv(DATA / "twitter_eda.csv")
df = pd.concat([unified, twitter], ignore_index=True)

print("unified_posts (YouTube + Instagram):", unified.shape)
print("twitter_eda:", twitter.shape)
print("combined:", df.shape)
df["Platform"].value_counts()

## 2. Dataset overview
Twitter dwarfs the clean platforms in volume, so the chart uses a log scale.

In [ ]:
counts = df["Platform"].value_counts()
ax = counts.plot(kind="bar", color=sns.color_palette("muted"))
ax.set_title("Records per Platform (log scale)")
ax.set_ylabel("records")
ax.set_yscale("log")
plt.tight_layout(); plt.savefig(FIG_DIR / "records_per_platform.png", dpi=120); plt.show()
counts

## 3. Audience interests - keyword volume (YouTube + Instagram)
Which topics attract the most content on the balanced platforms.

In [ ]:
clean = df[df["Platform"].isin(["youtube", "instagram"])]
pivot = clean.pivot_table(index="Keyword", columns="Platform",
                          values="Content", aggfunc="size", fill_value=0)
pivot = pivot.sort_values("youtube")
ax = pivot.plot(kind="barh")
ax.set_title("Audience Interest: Keyword Volume (YouTube + Instagram)")
ax.set_xlabel("records")
plt.tight_layout(); plt.savefig(FIG_DIR / "keyword_volume_clean.png", dpi=120); plt.show()
pivot

## 4. Twitter bias - AWS dominance
Why Twitter is EDA-only: ~99% of keyword-matched tweets are AWS, so it cannot represent the other topics or be used for balanced modeling.

In [ ]:
tw = twitter["Keyword"].value_counts()
ax = tw.plot(kind="barh", color="#c44e52")
ax.set_title("Twitter Keyword Distribution - AWS Bias")
ax.set_xlabel("tweets")
for i, v in enumerate(tw.values):
    ax.text(v, i, f" {v:,}", va="center")
plt.tight_layout(); plt.savefig(FIG_DIR / "twitter_aws_bias.png", dpi=120); plt.show()
print(f"AWS share of Twitter: {tw.get('AWS', 0) / tw.sum():.1%}")

## 5. Engagement patterns

In [ ]:
eng = df.groupby("Platform")["Engagement Score"].mean().sort_values()
ax = eng.plot(kind="bar", color=sns.color_palette("muted"))
ax.set_title("Average Engagement Score by Platform")
ax.set_ylabel("avg engagement")
plt.tight_layout(); plt.savefig(FIG_DIR / "avg_engagement_platform.png", dpi=120); plt.show()

keng = clean.groupby("Keyword")["Engagement Score"].mean().sort_values()
ax = keng.plot(kind="barh", color="#4c72b0")
ax.set_title("Average Engagement by Keyword (YouTube + Instagram)")
ax.set_xlabel("avg engagement")
plt.tight_layout(); plt.savefig(FIG_DIR / "engagement_by_keyword.png", dpi=120); plt.show()

## 6. Top authors by engagement (YouTube + Instagram)

In [ ]:
top = (clean.groupby(["Platform", "Author"])["Engagement Score"].sum()
       .sort_values(ascending=False).head(10).reset_index())
top

## 7. Export summary tables
Persist the key summary tables as CSVs for the report (replaces the old eda.py outputs).

In [ ]:
platform_summary = df.groupby("Platform").agg(
    records=("Content", "size"),
    keywords=("Keyword", "nunique"),
    avg_engagement=("Engagement Score", "mean"),
    median_engagement=("Engagement Score", "median"),
    max_engagement=("Engagement Score", "max"),
).round(2).reset_index()

keyword_distribution = (df.groupby(["Platform", "Keyword"]).size()
                        .reset_index(name="records")
                        .sort_values(["Platform", "records"], ascending=[True, False]))

engagement_by_keyword = df.groupby(["Platform", "Keyword"]).agg(
    records=("Engagement Score", "size"),
    avg_engagement=("Engagement Score", "mean"),
    max_engagement=("Engagement Score", "max"),
).round(2).reset_index()

for name, table in {
    "platform_summary": platform_summary,
    "keyword_distribution": keyword_distribution,
    "engagement_by_keyword": engagement_by_keyword,
    "top_authors": top,
}.items():
    table.to_csv(REPORT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
    print("saved", name, table.shape)

platform_summary

## Key takeaways

- YouTube + Instagram are the analysis/modeling-ready sources, balanced across all 12 keywords and temporally current.
- Twitter is documented as EDA-only due to a ~99% AWS bias and pre-ChatGPT timeframe; isolating it in `twitter_eda` prevents bias leaking into modeling.
- Engagement is platform-specific, so compare within a platform, not across (raw scores differ in meaning).
- Next phase: sentiment analysis (YouTube/Instagram need a model; Twitter has a pre-computed sentiment score) and topic modeling.